In [1]:
import torch
import matplotlib.pyplot as plt
import pickle
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from tools import Dictionary
from GloVe import GloVeModel

In [2]:
EMBEDDING_SIZE = 200
CONTEXT_SIZE = 5
MODEL_PATH = './glove.pt'
DOC_PATH = './corpus.pickle'
COMATRIX_PATH = './comat.pickle'
TXT_EMBED_PATH = './glove_custom_200d.txt'

In [3]:
def load_model_and_dictionary():
    with open(DOC_PATH, 'rb') as f:
        doc = pickle.load(f)
    dictionary = Dictionary(doc)
    vocab_size = dictionary.vocab_size

    model = GloVeModel(EMBEDDING_SIZE, CONTEXT_SIZE, vocab_size)
    model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))

    return model, dictionary

def visualize_loss(train_losses, val_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss over Epochs")
    plt.legend()
    plt.grid(True)
    plt.show()

def get_embeddings(model):
    focal = model._focal_embeddings.weight.data.cpu().numpy()
    context = model._context_embeddings.weight.data.cpu().numpy()
    return (focal + context) / 2

def visualize_embeddings_2d(embeddings, dictionary, method='pca', num_words=200):
    words = list(dictionary.word2idx.keys())[:num_words]
    idx = [dictionary.word2idx[word] for word in words]
    vectors = embeddings[idx]

    if method == 'pca':
        reducer = PCA(n_components=2)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, perplexity=30, random_state=42)
    else:
        raise ValueError("method must be 'pca' or 'tsne'")

    reduced = reducer.fit_transform(vectors)

    plt.figure(figsize=(12, 10))
    for i, word in enumerate(words):
        x, y = reduced[i]
        plt.scatter(x, y)
        plt.text(x + 0.01, y + 0.01, word, fontsize=9)
    plt.title(f"Word Embeddings projected to 2D using {method.upper()}")
    plt.grid(True)
    plt.show()

def find_similar_words(target_word, embeddings, dictionary, top_k=10):
    word2idx = dictionary.word2idx
    if target_word not in word2idx:
        print(f"{target_word} not found in vocabulary.")
        return

    idx2word = {idx: word for word, idx in word2idx.items()}
    target_vec = embeddings[word2idx[target_word]]
    norm_embeds = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    target_vec = target_vec / np.linalg.norm(target_vec)

    similarities = np.dot(norm_embeds, target_vec)
    best_idx = np.argsort(-similarities)[1:top_k + 1]

    print(f"\nTop {top_k} words similar to '{target_word}':")
    for i in best_idx:
        print(f"{idx2word[i]} - similarity: {similarities[i]:.4f}")

In [4]:
cyberbullying_keywords = [
    "hate", "stupid", "ugly", "idiot", "loser", "fat", "kill", "worthless",
    "bitch", "slut", "freak", "retard", "dumb", "disgusting", "annoying",
    "trash", "moron", "jerk", "hoe", "pathetic"
]

In [5]:
model, dictionary = load_model_and_dictionary()
embeddings = get_embeddings(model)

In [6]:
for word in cyberbullying_keywords:
    find_similar_words(word, embeddings, dictionary, top_k=5)


Top 5 words similar to 'hate':
manati - similarity: 0.3728
beads - similarity: 0.3367
thinks - similarity: 0.3288
presley - similarity: 0.3103
repeatedly - similarity: 0.2921

Top 5 words similar to 'stupid':
headbanging - similarity: 0.4191
heresy - similarity: 0.3700
contemplative - similarity: 0.2583
fringe - similarity: 0.2569
dem - similarity: 0.2441

Top 5 words similar to 'ugly':
shove - similarity: 0.3367
demonstrates - similarity: 0.3325
cimicifugia - similarity: 0.3214
acid - similarity: 0.2778
morphs - similarity: 0.2696

Top 5 words similar to 'idiot':
calvinists - similarity: 0.3415
phallic - similarity: 0.3181
existem - similarity: 0.3134
laughs - similarity: 0.2963
screencaps - similarity: 0.2954

Top 5 words similar to 'loser':
kazakhistan - similarity: 0.3841
artifacts - similarity: 0.2853
christina - similarity: 0.2828
createdtook - similarity: 0.2824
hatsune - similarity: 0.2689

Top 5 words similar to 'fat':
edgbaston - similarity: 0.5223
gestapo - similarity: 0.28